<a href="https://colab.research.google.com/github/kursatkara/MAE_5020_Spring_2026/blob/master/06_Notebook_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook A — Verifying Backpropagation for a Simple Neural Network

## 🎯 Learning Objective
Understand and **verify** one forward and backward pass for a simple neural network using:
- manual derivation
- PyTorch autograd

> Key idea:
> $$
\hat{y} = f(x; \theta), \quad \min_{\theta} L(\theta)
$$

---

## 🧠 Challenge

Before trusting automatic differentiation:

> **Do the gradients computed by PyTorch match the gradients we derive by hand?**



## 🔹 Model Definition

We consider a 1-1-1 neural network:

$$
z^1 = W^1 x + b^1
$$
$$
a^1 = \sigma(z^1)
$$
$$
z^2 = W^2 a^1 + b^2
$$
$$
\hat{y} = \sigma(z^2)
$$

Loss:
$$
L = \frac{1}{2}(\hat{y} - y)^2
$$

---

### Dimensions

- $x \in \mathbb{R}$
- $W^1, W^2 \in \mathbb{R}$
- $b^1, b^2 \in \mathbb{R}$

👉 Everything is scalar → easier to verify.



## 🔹 Understanding the Notation

We consider a **1–1–1 neural network**, which means:

- **1 input**
- **1 hidden neuron**
- **1 output**

This is the smallest neural network that still contains all of the essential ideas:
- a linear transformation,
- a nonlinear activation,
- a second linear transformation,
- a prediction,
- a loss function.

### Simple diagram

```text
input x  ──(W^1, b^1)──>  z^1  ──σ──>  a^1  ──(W^2, b^2)──>  z^2  ──σ──>  ŷ
```

You can read this from left to right:

1. start with the input $x$,
2. apply a linear transformation to get $z^1$,
3. apply the activation function to get $a^1$,
4. apply another linear transformation to get $z^2$,
5. apply the activation again to get the prediction $\hat{y}$.

---

### Step-by-step meaning

#### Input
- $x$ = input value

#### Hidden-layer pre-activation
$$
z^1 = W^1 x + b^1
$$

- $W^1$ = weight connecting the input to the hidden neuron
- $b^1$ = bias of the hidden neuron
- $z^1$ = hidden-layer pre-activation, meaning the value **before** applying the nonlinearity

At this stage, the model is still linear in $x$.

#### Hidden-layer activation
$$
a^1 = \sigma(z^1)
$$

- $\sigma(\cdot)$ = activation function
- $a^1$ = hidden-layer activation, meaning the value **after** applying the nonlinearity

This is where the network becomes nonlinear.

#### Output-layer pre-activation
$$
z^2 = W^2 a^1 + b^2
$$

- $W^2$ = weight connecting the hidden neuron to the output
- $b^2$ = output bias
- $z^2$ = output-layer pre-activation

#### Network prediction
$$
\hat{y} = \sigma(z^2)
$$

- $\hat{y}$ (“y-hat”) = predicted output of the neural network

---

### Loss function

We compare the prediction $\hat{y}$ to the true target value $y$ using the loss
$$
L = \frac{1}{2}(\hat{y} - y)^2.
$$

Here:
- $y$ = true target value
- $\hat{y}$ = predicted value
- $L$ = loss, or error measure

The squared term penalizes larger errors more strongly.

The factor $\frac{1}{2}$ is included for convenience, because when we differentiate the loss with respect to $\hat{y}$, we get
$$
\frac{\partial L}{\partial \hat{y}} = \hat{y} - y,
$$
which keeps the gradient expressions cleaner.

---

### Big-picture interpretation

The full network is a parameterized function:
$$
\hat{y} = f(x; \theta),
$$
where the trainable parameters are
$$
\theta = \{W^1, b^1, W^2, b^2\}.
$$

Even though this example is tiny, it contains the essential ingredients of a neural network:
- linear maps,
- nonlinear activations,
- trainable parameters,
- a loss function,
- gradient-based learning.

That is why this small example is so useful pedagogically.


In [1]:

import torch
import numpy as np

torch.manual_seed(0)

x = torch.tensor([1.0])
y = torch.tensor([0.0])

W1 = torch.tensor([[0.5]], requires_grad=True)
b1 = torch.tensor([0.0], requires_grad=True)
W2 = torch.tensor([[0.5]], requires_grad=True)
b2 = torch.tensor([0.0], requires_grad=True)

sigmoid = torch.sigmoid


## 🔹 Forward Pass (Step-by-Step)

In [2]:

# Step 1: Hidden layer pre-activation
z1 = W1 @ x + b1

# Step 2: Activation
a1 = sigmoid(z1)

# Step 3: Output layer
z2 = W2 @ a1 + b2
y_hat = sigmoid(z2)

# Step 4: Loss
loss = 0.5 * (y_hat - y)**2

print("z1:", z1.item())
print("a1:", a1.item())
print("z2:", z2.item())
print("y_hat:", y_hat.item())
print("loss:", loss.item())


z1: 0.5
a1: 0.622459352016449
z2: 0.3112296760082245
y_hat: 0.5771853923797607
loss: 0.1665714830160141



## 🔹 Backward Pass (Autograd)

We compute gradients:
$$
\frac{\partial L}{\partial W^1}, \quad \frac{\partial L}{\partial W^2}
$$

Expected chain rule structure:
$$
\frac{\partial L}{\partial W^2} = (\hat{y}-y)\sigma'(z^2)a^1
$$


In [3]:

loss.backward()

grad_W1 = W1.grad.clone()
grad_W2 = W2.grad.clone()

print("grad_W1:", grad_W1.item())
print("grad_W2:", grad_W2.item())


grad_W1: 0.01655104197561741
grad_W2: 0.0876782014966011



## 🔹 Verification via Finite Differences

We approximate gradients numerically:
$$
\frac{\partial L}{\partial W} \approx \frac{L(W+\epsilon) - L(W-\epsilon)}{2\epsilon}
$$

This provides an independent check.


In [4]:

def compute_loss(W1_val):
    z1 = W1_val @ x + b1
    a1 = sigmoid(z1)
    z2 = W2 @ a1 + b2
    y_hat = sigmoid(z2)
    return 0.5*(y_hat - y)**2

eps = 1e-5

W1_plus = W1.detach().clone() + eps
W1_minus = W1.detach().clone() - eps

fd_grad = (compute_loss(W1_plus) - compute_loss(W1_minus)) / (2*eps)

print("Finite difference grad W1:", fd_grad.item())
print("Autograd grad W1:", grad_W1.item())


Finite difference grad W1: 0.017136335372924805
Autograd grad W1: 0.01655104197561741



## 🔹 Reflection Questions

1. Which gradient is larger: $W^1$ or $W^2$?
2. Why does the chain rule amplify or dampen gradients?
3. What would happen if $\sigma$ were linear?

---

## 🔍 Sanity Check

- Are gradients nonzero?
- Does the sign of the gradient make sense?
- Would gradient descent reduce the loss?

---

## 🧠 Engineering Takeaway

> Backpropagation is **structured sensitivity analysis**:
$$
\nabla_\theta L
$$

This is the same idea used in:
- system identification
- adjoint methods
- optimization in CFD

---

## 🚀 Next

In the next notebook, we will **repeat this update many times** and study training as a dynamical system.
